In [2]:
VISION_AGENT_API_KEY="dWRqMDY0ZDlhOTdvZXU5ODM5bThqOllaT1FUSXRiaDN1d3JzMnRLamwzTWZPeXVTOHlUbUZl"

In [ ]:
import os
import re
import json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import fitz  
from pymupdf4llm import to_markdown
from landingai_ade import LandingAIADE

INPUT_DIR = Path("input_files")
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)
API_KEY = VISION_AGENT_API_KEY
NUM_WORKERS = 10

client = LandingAIADE(apikey=API_KEY)

def clean_markdown(md_text: str) -> str:
    md_text = re.sub(r"<::.*?::>", "", md_text, flags=re.DOTALL)
    md_text = re.sub(r"<a id=['\"].*?['\"]></a>", "", md_text)

    def html_table_to_text(match):
        table_html = match.group(0)
        rows = re.findall(r"<tr>(.*?)</tr>", table_html, flags=re.DOTALL)
        table_text = []
        for r in rows:
            cells = re.findall(r"<t[dh][^>]*>(.*?)</t[dh]>", r, flags=re.DOTALL)
            row_text = " | ".join(cell.strip() for cell in cells)
            table_text.append(row_text)
        return "\n".join(table_text)

    md_text = re.sub(r"<table.*?>.*?</table>", html_table_to_text, md_text, flags=re.DOTALL)

    html_unescape = {"&lt;": "<", "&gt;": ">", "&amp;": "&", "&quot;": '"', "&#39;": "'"}
    for k, v in html_unescape.items():
        md_text = md_text.replace(k, v)

    md_text = re.sub(r"\n{3,}", "\n\n", md_text).strip()

    return md_text

def has_text_layer(pdf_path: Path) -> bool:
    try:
        with fitz.open(pdf_path) as doc:
            for page in doc:
                text = page.get_text("text").strip()
                if text:
                    return True
    except Exception:
        return False
    return False

def process_single_file(file_path: Path):
    try:
        text_result = ""

        if file_path.suffix.lower() == ".pdf" and has_text_layer(file_path):
            md_text = to_markdown(file_path)
            text_result = clean_markdown(md_text)
            method = "PyMuPDF4LLM"
        else:
            response = client.parse(document=file_path)
            md_text = response.markdown
            text_result = clean_markdown(md_text)
            method = "ADE OCR"

        out_txt = OUTPUT_DIR / f"{file_path.stem}.txt"
        out_txt.write_text(text_result, encoding="utf-8")

        return f"{file_path.name} ({method})"
    except Exception as e:
        return f" {file_path.name}: {e}"

def process_folder(input_dir: Path):
    files = [f for f in input_dir.iterdir() if f.suffix.lower() in [".pdf", ".jpg", ".jpeg", ".png", ".tiff", ".webp"]]
    
    results = []
    with ThreadPoolExecutor(max_workers=NUM_WORKERS) as executor:
        futures = {executor.submit(process_single_file, f): f for f in files}
        for future in tqdm(as_completed(futures), total=len(futures)):
            results.append(future.result())

    print("\n".join(results))

if __name__ == "__main__":
    process_folder(INPUT_DIR)


100%|██████████| 8/8 [00:11<00:00,  1.45s/it]

pdf_text.pdf (PyMuPDF4LLM)
QCDT-2023-upload.pdf (PyMuPDF4LLM)
scan_src.png (ADE OCR)
text_image.jpg (ADE OCR)
table.png (ADE OCR)
1.webp (ADE OCR)
pdf_scan.pdf (ADE OCR)
2.webp (ADE OCR)
